# Flower Recognition — Training (Oxford 102 Flowers)

Fine-tunes an ImageNet-pretrained backbone (via `timm`) on the Oxford 102
Flowers dataset, evaluates it, and exports it to ONNX for the deployed web
app. Designed to run for free on **Colab's T4 GPU**: Runtime -> Change
runtime type -> T4 GPU.

If Colab's free GPU quota is unavailable on a given day, [Kaggle
Notebooks](https://www.kaggle.com/code) gives 30 free GPU-hours/week (T4/P100)
as a documented backup — the training scripts here are plain PyTorch and work
identically there.

**Before running**: push this repo to GitHub first, then replace `REPO_URL`
below with your repo's clone URL.

## 1. Clone the repo and install extra dependencies

`torch`/`torchvision` are already GPU-matched and preinstalled on Colab — do **not** reinstall them, just add the libraries this project needs on top.

In [1]:
REPO_URL = "https://github.com/hindh00/flower-recognition.git"  # <-- replace

!git clone $REPO_URL repo
%cd repo/training

!pip install -q timm grad-cam onnx onnxruntime scikit-learn


Cloning into 'repo'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 61 (delta 6), reused 55 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 88.26 KiB | 11.03 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/repo/training
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 31.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generative

## 2. Download the dataset

Downloads Oxford 102 Flowers via `torchvision` (no auth needed). If you see an `SSL: CERTIFICATE_VERIFY_FAILED` error, `dataset.py` automatically retries with an unverified context — this is a known, documented flakiness of `robots.ox.ac.uk` from hosted notebooks, not a bug in this code.

Optional: mount Google Drive first and point `DATA_ROOT` at it so the ~330MB download is cached across sessions instead of re-downloading every time you reconnect.

In [ ]:
# cache the dataset on Drive across sessions.
from google.colab import drive
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/flowers102_data"

from dataset import load_all_samples

samples = load_all_samples(DATA_ROOT)
print(f"Total images across all official splits: {len(samples)}")


100%|██████████| 345M/345M [00:24<00:00, 14.2MB/s]
100%|██████████| 502/502 [00:00<00:00, 513kB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 41.1MB/s]

Total images across all official splits: 8189


## 3. Quick EDA

Class counts and image-size distribution — this is what sets up the class-imbalance discussion in the README (some classes have far fewer images than others).

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image
import random

labels = [label for _, label in samples]
counts = Counter(labels)
print(f"Classes: {len(counts)}  min/class: {min(counts.values())}  max/class: {max(counts.values())}  "
      f"mean/class: {sum(counts.values()) / len(counts):.1f}")

plt.figure(figsize=(14, 4))
plt.bar(range(len(counts)), [counts[i] for i in range(len(counts))])
plt.xlabel("Class index")
plt.ylabel("Image count")
plt.title("Images per class (Oxford 102 Flowers, merged splits)")
plt.tight_layout()
plt.savefig("../docs/images/class_distribution.png", dpi=150)
plt.show()

# A handful of sample images
sample_paths = random.sample(samples, 6)
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, (path, label) in zip(axes, sample_paths):
    ax.imshow(Image.open(path))
    ax.set_title(f"class {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Train

Two-stage fine-tune: frozen-backbone warm-up, then full unfreeze with a cosine-annealed LR. ~1-3 min/epoch for EfficientNetB0 at 224px on a T4, so the full run below is comfortably under an hour.

In [ ]:
from train import train

checkpoint_path = train(
    data_root=DATA_ROOT,
    backbone="efficientnet_b0",
    warmup_epochs=3,
    finetune_epochs=15,
    batch_size=32,
    out_dir="checkpoints",
)


## 5. Evaluate

Test-set top-1/top-5 accuracy, a per-class classification report, a confusion-matrix heatmap, and a "most confused pairs" table. Writes `docs/results.md` and `docs/images/confusion_matrix.png` directly into the repo so they're ready to commit.

In [ ]:
from evaluate import evaluate

top1, top5 = evaluate(
    data_root=DATA_ROOT,
    checkpoint=str(checkpoint_path),
    backbone="efficientnet_b0",
    cat_to_name_path="cat_to_name.json",
    out_dir="../docs",
)
print(f"Top-1: {top1:.4f}   Top-5: {top5:.4f}")


## 6. Grad-CAM (optional, stretch goal)

Visualize what the model is looking at for a few test images — pick a couple of correct predictions and a couple of mistakes for the most interesting README material. Grab some real test-image paths/labels from `samples` above.

In [ ]:
from grad_cam import run_gradcam

# Replace with a handful of real (path, label) pairs from `samples`, ideally
# including at least one misclassified example found during evaluation.
example_paths = [str(p) for p, _ in samples[:4]]
example_labels = [l for _, l in samples[:4]]

run_gradcam(
    example_paths,
    example_labels,
    checkpoint=str(checkpoint_path),
    backbone="efficientnet_b0",
    cat_to_name_path="cat_to_name.json",
    out_dir="../docs/images",
)


## 7. Export to ONNX

Exports the fine-tuned model for the deployed API and runs a PyTorch-vs-ONNXRuntime parity check — if this assertion fails, do not ship the exported model.

In [ ]:
from export_onnx import export

export(
    checkpoint=str(checkpoint_path),
    backbone="efficientnet_b0",
    cat_to_name_path="cat_to_name.json",
    out_path="../api/model/flower_model.onnx",
)


## 8. Get the results back into your repo

The exported files now live at `api/model/flower_model.onnx` and
`api/model/class_names.json`, and the evaluation artifacts are in `docs/`.
From this Colab session:

```python
%cd /content/repo
!git add api/model/flower_model.onnx api/model/class_names.json docs/results.md docs/images
!git commit -m "Add trained model and evaluation results"
!git push
```

You'll need to authenticate the push (a GitHub personal access token works
as the password prompt, or set up `git config` with credentials first).
Once pushed, Vercel's auto-deploy picks it up and the live demo starts
serving the trained model.